# Dataset Inspection

This notebook performs a lightweight inspection of the TruckScenes dataset before the main velocity and RADAR–LiDAR comparison analyses.

The inspection focuses on:

- available RADAR and LiDAR sensor channels
- measurement fields provided by each modality
- metadata required for vehicle-level velocity analysis
- temporal observation frequency of RADAR and LiDAR

The purpose is to confirm that the information required by the later analysis is available. Detailed sensor-performance analysis is performed in later notebooks.

The mini dataset can be used during development. The same code structure is intended to be reusable with the complete TruckScenes dataset.

In [ ]:
# Dataset Path and Setup
from pathlib import Path
import json
import pandas as pd

# Dataset configuration
DATA_ROOT = Path("../data/man-truckscenes")

SENSOR_ROOT = DATA_ROOT / "man-truckscenes"
SAMPLES_ROOT = SENSOR_ROOT / "samples"
SWEEPS_ROOT = SENSOR_ROOT / "sweeps"

# Change this when using the complete dataset
METADATA_DIR_NAME = "v1.2-mini"
METADATA_ROOT = DATA_ROOT / METADATA_DIR_NAME


print("Dataset:", METADATA_DIR_NAME)
print("Samples:", SAMPLES_ROOT.exists())
print("Sweeps:", SWEEPS_ROOT.exists())
print("Metadata:", METADATA_ROOT.exists())

Dataset: v1.2-mini
Samples: True
Sweeps: True
Metadata: True


## 1 Sensor Structure

The available RADAR and LiDAR sensor channels are identified from the sensor-data directory.

This establishes which sensor channels are available for later analysis.

In [37]:
sensor_records = []

for sensor_folder in sorted(SAMPLES_ROOT.iterdir()):

    if not sensor_folder.is_dir():
        continue

    sensor_name = sensor_folder.name

    if sensor_name.startswith("LIDAR"):
        modality = "LiDAR"

    elif sensor_name.startswith("RADAR"):
        modality = "RADAR"

    else:
        continue

    sensor_records.append({
        "Modality": modality,
        "Sensor": sensor_name,
        "Sample_PCD_Files": len(list(sensor_folder.glob("*.pcd"))),
    })


sensor_structure = pd.DataFrame(sensor_records)

display(sensor_structure)

,Modality,Sensor,Sample_PCD_Files
0,LiDAR,LIDAR_LEFT,400
1,LiDAR,LIDAR_REAR,400
2,LiDAR,LIDAR_RIGHT,400
3,LiDAR,LIDAR_TOP_FRONT,400
4,LiDAR,LIDAR_TOP_LEFT,400
5,LiDAR,LIDAR_TOP_RIGHT,400
6,RADAR,RADAR_LEFT_BACK,400
7,RADAR,RADAR_LEFT_FRONT,400
8,RADAR,RADAR_LEFT_SIDE,400
9,RADAR,RADAR_RIGHT_BACK,400


## 2 RADAR and LiDAR Measurement Fields

A representative PCD file from each sensor channel is inspected to identify the measurements provided by RADAR and LiDAR.

This inspection is intended to verify the sensor-data structure rather than analyse every PCD file individually.

In [38]:
# header reader
def read_pcd_header(pcd_file):
    header = {}

    with open(pcd_file, "rb") as f:

        while True:

            line = f.readline()

            if not line:
                break

            text = line.decode(
                "utf-8",
                errors="ignore"
            ).strip()

            if text.startswith("FIELDS"):
                header["FIELDS"] = text.replace("FIELDS ", "")

            elif text.startswith("POINTS"):
                header["POINTS"] = int(text.split()[1])

            elif text.startswith("DATA"):
                header["DATA"] = text.replace("DATA ", "")
                break

    return header


field_records = []

for sensor_folder in sorted(SAMPLES_ROOT.iterdir()):

    if not sensor_folder.is_dir():
        continue

    sensor_name = sensor_folder.name

    if sensor_name.startswith("LIDAR"):
        modality = "LiDAR"

    elif sensor_name.startswith("RADAR"):
        modality = "RADAR"

    else:
        continue

    pcd_files = sorted(sensor_folder.glob("*.pcd"))

    if not pcd_files:
        continue

    example_file = pcd_files[0]

    header = read_pcd_header(example_file)

    field_records.append({
        "Modality": modality,
        "Sensor": sensor_name,
        "Fields": header.get("FIELDS"),
        "Example_Points": header.get("POINTS"),
    })


field_summary = pd.DataFrame(field_records)

display(field_summary)

,Modality,Sensor,Fields,Example_Points
0,LiDAR,LIDAR_LEFT,x y z intensity timestamp,80729
1,LiDAR,LIDAR_REAR,x y z intensity timestamp,17262
2,LiDAR,LIDAR_RIGHT,x y z intensity timestamp,81221
3,LiDAR,LIDAR_TOP_FRONT,x y z intensity timestamp,17458
4,LiDAR,LIDAR_TOP_LEFT,x y z intensity timestamp,23344
5,LiDAR,LIDAR_TOP_RIGHT,x y z intensity timestamp,22034
6,RADAR,RADAR_LEFT_BACK,x y z vrel_x vrel_y vrel_z rcs,677
7,RADAR,RADAR_LEFT_FRONT,x y z vrel_x vrel_y vrel_z rcs,612
8,RADAR,RADAR_LEFT_SIDE,x y z vrel_x vrel_y vrel_z rcs,416
9,RADAR,RADAR_RIGHT_BACK,x y z vrel_x vrel_y vrel_z rcs,430


### 2.1 Field Summary

The inspected LiDAR channels provide:

- `x`, `y`, `z`
- `intensity`
- `timestamp`

The inspected RADAR channels provide:

- `x`, `y`, `z`
- `vrel_x`, `vrel_y`, `vrel_z`
- `rcs`

RADAR therefore provides direct point-level relative velocity measurements.

LiDAR does not contain a direct velocity field, so vehicle-level LiDAR velocity must later be estimated using temporal information.

## 3 Required Metadata

The metadata required for temporal alignment, object tracking, reference vehicle-velocity calculation, ego motion, and sensor calibration is checked before the main analysis.

In [39]:
required_metadata = {

    "sample.json": [
        "token",
        "timestamp",
        "scene_token",
    ],

    "sample_data.json": [
        "token",
        "sample_token",
        "calibrated_sensor_token",
        "timestamp",
        "filename",
    ],

    "sample_annotation.json": [
        "token",
        "sample_token",
        "instance_token",
        "translation",
        "prev",
        "next",
    ],

    "instance.json": [
        "token",
        "category_token",
    ],

    "ego_pose.json": [
        "token",
        "translation",
        "rotation",
    ],

    "calibrated_sensor.json": [
        "token",
        "sensor_token",
        "translation",
        "rotation",
    ],

    "sensor.json": [
        "token",
        "channel",
        "modality",
    ],

    "ego_motion_chassis.json": [
        "timestamp",
        "vx",
        "vy",
        "vz",
    ],
}


metadata_records = []

for filename, required_fields in required_metadata.items():

    file_path = METADATA_ROOT / filename

    if not file_path.exists():

        metadata_records.append({
            "Metadata_File": filename,
            "Records": 0,
            "Required_Fields_Available": False,
        })

        continue


    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    records = data if isinstance(data, list) else [data]


    fields_available = all(
        all(field in record for field in required_fields)
        for record in records
    )


    metadata_records.append({
        "Metadata_File": filename,
        "Records": len(records),
        "Required_Fields_Available": fields_available,
    })


metadata_summary = pd.DataFrame(metadata_records)

display(metadata_summary)

,Metadata_File,Records,Required_Fields_Available
0,sample.json,400,True
1,sample_data.json,43556,True
2,sample_annotation.json,25750,True
3,instance.json,1094,True
4,ego_pose.json,20116,True
5,calibrated_sensor.json,18,True
6,sensor.json,18,True
7,ego_motion_chassis.json,20089,True


### 3.1 Metadata Summary

The required metadata is available for the planned analysis.

In particular:

- timestamps support temporal alignment
- `instance_token`, `prev`, and `next` support object tracking
- annotation `translation` can be used to derive reference vehicle velocity
- ego-motion metadata provides ego-vehicle velocity
- calibrated-sensor metadata provides sensor position and orientation
- sensor metadata identifies sensor channels and modalities

These relationships are used directly in the later velocity-analysis notebooks.

## 4 Temporal Observation Frequency

RADAR and LiDAR do not necessarily provide observations at the same temporal frequency.

To support later comparison over equivalent observation periods, sensor timestamps are analysed within individual scenes.

Intervals are calculated within scenes to avoid introducing artificial gaps between separate scenes.

In [40]:
# Load metadata
with open(METADATA_ROOT / "sample.json", "r", encoding="utf-8") as f:
    samples = json.load(f)

with open(METADATA_ROOT / "sample_data.json", "r", encoding="utf-8") as f:
    sample_data = json.load(f)

with open(METADATA_ROOT / "calibrated_sensor.json", "r", encoding="utf-8") as f:
    calibrated_sensors = json.load(f)

with open(METADATA_ROOT / "sensor.json", "r", encoding="utf-8") as f:
    sensors = json.load(f)

# Lookup tables
sample_to_scene = {
    record["token"]: record["scene_token"]
    for record in samples
}

calibrated_sensor_lookup = {
    record["token"]: record
    for record in calibrated_sensors
}

sensor_lookup = {
    record["token"]: record
    for record in sensors
}

# RADAR / LiDAR observations
sensor_records = []

for record in sample_data:

    calibrated_sensor = calibrated_sensor_lookup.get(
        record["calibrated_sensor_token"]
    )

    if calibrated_sensor is None:
        continue

    sensor = sensor_lookup.get(
        calibrated_sensor["sensor_token"]
    )

    if sensor is None:
        continue

    modality_raw = sensor.get("modality", "").lower()

    if modality_raw == "lidar":
        modality = "LiDAR"

    elif modality_raw == "radar":
        modality = "RADAR"

    else:
        continue


    sensor_records.append({
        "Modality": modality,
        "Sensor": sensor["channel"],
        "Scene_Token": sample_to_scene.get(
            record["sample_token"]
        ),
        "Timestamp": record["timestamp"],
    })


sensor_data_df = pd.DataFrame(sensor_records)

# Within-scene intervals
interval_records = []

for (modality, sensor, scene), group in sensor_data_df.groupby(
    ["Modality", "Sensor", "Scene_Token"]
):

    timestamps = (
        group["Timestamp"]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .to_numpy()
    )

    if len(timestamps) < 2:
        continue

    intervals = timestamps[1:] - timestamps[:-1]

    for interval in intervals:

        interval_records.append({
            "Modality": modality,
            "Sensor": sensor,
            "Interval": interval,
        })


interval_df = pd.DataFrame(interval_records)

# Sensor-level frequency
sensor_frequency = (
    interval_df
    .groupby(["Modality", "Sensor"])
    .agg(
        Median_Interval=("Interval", "median")
    )
    .reset_index()
)

sensor_frequency["Approx_Frequency_Hz"] = (
    1_000_000
    / sensor_frequency["Median_Interval"]
)

# Modality-level summary
frequency_summary = (
    sensor_frequency
    .groupby("Modality")
    .agg(
        Median_Timestamp_Interval=(
            "Median_Interval",
            "median"
        ),
        Approx_Frequency_Hz=(
            "Approx_Frequency_Hz",
            "median"
        ),
    )
    .round(2)
)


display(frequency_summary)

,Median_Timestamp_Interval,Approx_Frequency_Hz
Modality,,
LiDAR,99989.5,10.0
RADAR,51278.0,19.5


### 4.1 Temporal Summary

The within-scene timestamp analysis shows that LiDAR observations occur at approximately 10 Hz, while RADAR observations occur at approximately 19.5 Hz in TruckScenes v1.2-mini.

RADAR therefore provides sensor observations at a higher temporal frequency than LiDAR.

Temporal observation frequency is relevant to later information-acquisition analysis, but a higher observation frequency alone does not indicate better overall sensor performance.

## 5 Key Variable Definitions

The following variables are relevant to the later velocity, temporal, object-level, and sensor-comparison analyses.

| Variable | Source | Meaning | Unit | Coordinate / Reference Frame |
|---|---|---|---|---|
| `x`, `y`, `z` | RADAR PCD | Position of each RADAR return | m | RADAR sensor frame |
| `vrel_x`, `vrel_y`, `vrel_z` | RADAR PCD | Relative velocity components associated with each RADAR return | m/s | RADAR sensor frame |
| `rcs` | RADAR PCD | Radar cross-section measurement associated with each RADAR return | To be verified | Sensor measurement |
| `x`, `y`, `z` | LiDAR PCD | Position of each LiDAR point | m | LiDAR sensor frame |
| `intensity` | LiDAR PCD | LiDAR return-intensity measurement | To be verified | Sensor measurement |
| `timestamp` | LiDAR PCD | Timestamp associated with each LiDAR point | To be verified | — |
| `vx`, `vy`, `vz` | `ego_motion_chassis.json` | Ego-vehicle velocity components | m/s | Vehicle frame |
| `translation` | `sample_annotation.json` | Centre position of an annotated 3D bounding box | m | Global frame |
| `translation` | `ego_pose.json` | Ego-vehicle position | m | Global frame |
| `timestamp` | `sample.json` / `sample_data.json` | Timestamp associated with a sample or sensor observation | Time unit used by dataset metadata | — |
| `instance_token` | `sample_annotation.json` | Identifier linking annotations belonging to the same physical object | — | — |
| `prev`, `next` | `sample_annotation.json` | Links to previous and next records in a temporal sequence | — | — |
| `sensor_token` | `calibrated_sensor.json` | Identifier linking a calibrated sensor to its sensor definition | — | — |
| `channel` | `sensor.json` | Sensor-channel name | — | — |
| `modality` | `sensor.json` | Sensor modality such as RADAR or LiDAR | — | — |

## 6 Evidence Summary

| Topic | Finding | Source | Verified locally? |
|---|---|---|---|
| Sensor channels | Six LiDAR and six RADAR channels are present in the inspected `samples` directory | `samples/` | Yes |
| Sample PCD availability | Each inspected RADAR and LiDAR channel contains 400 sample PCD files | `samples/` | Yes |
| PCD consistency | All 4,800 inspected sample PCD files use a consistent field structure within their respective sensor channels | PCD headers | Yes |
| RADAR fields | RADAR PCD files contain `x`, `y`, `z`, `vrel_x`, `vrel_y`, `vrel_z`, and `rcs` | RADAR PCD headers | Yes |
| LiDAR fields | LiDAR PCD files contain `x`, `y`, `z`, `intensity`, and `timestamp` | LiDAR PCD headers | Yes |
| Direct RADAR velocity | Relative velocity components are directly available at the RADAR-return level | RADAR PCD headers | Yes |
| Direct LiDAR velocity | No direct velocity field is present in the inspected LiDAR PCD files | LiDAR PCD headers | Yes |
| Point observations | LiDAR PCD files contain substantially more point observations per file than RADAR PCD files | All 4,800 sample PCD headers | Yes |
| Metadata availability | All metadata files required for the planned analysis were found | `v1.2-mini/` | Yes |
| Timestamp information | Sample and sensor-observation metadata contain timestamps | `sample.json`, `sample_data.json` | Yes |
| Object tracking | Annotation metadata contains `instance_token`, `prev`, and `next` information | `sample_annotation.json`, `instance.json` | Yes |
| Object position | Annotation metadata provides object-centre `translation` | `sample_annotation.json` | Yes |
| Direct object velocity | No direct object-level velocity field was identified in the inspected annotation metadata | `sample_annotation.json` | Yes |
| Ego motion | Ego-motion metadata contains `vx`, `vy`, and `vz` | `ego_motion_chassis.json` | Yes |
| Sensor calibration | Sensor translation and rotation are available | `calibrated_sensor.json` | Yes |
| Intermediate observations | Both modalities contain sweep observations in addition to key samples | `sample_data.json` | Yes |
| Temporal frequency | RADAR observations occur at a higher temporal frequency than LiDAR observations in the inspected mini dataset | Within-scene `sample_data.json` timestamps | Yes |
| Approximate LiDAR frequency | LiDAR sensor channels operate at approximately 10 Hz based on median within-scene timestamp intervals | `sample_data.json` | Yes |
| Approximate RADAR frequency | RADAR sensor channels operate at approximately 19.5 Hz based on median within-scene timestamp intervals | `sample_data.json` | Yes |

## 7 Limitations

This notebook verifies the structure and availability of information in TruckScenes v1.2-mini but does not evaluate the overall performance of RADAR or LiDAR.

Several limitations should be considered:

- Point count represents the number of sensor returns or observations, but it does not directly represent the total amount or usefulness of sensor information.
- RADAR and LiDAR provide different measurement fields, so their information content cannot be compared using point count alone.
- The higher temporal observation frequency of RADAR does not by itself indicate greater overall information-acquisition efficiency.
- Vehicle-level sensor information has not yet been established because sensor returns have not yet been associated with individual annotated vehicles.
- The relationship between sensor information, vehicle speed, object distance, and object type has not yet been evaluated.
- RADAR relative velocity measurements are available at the point level and cannot yet be treated directly as vehicle-level velocity.
- LiDAR does not contain a direct velocity field, so vehicle velocity must later be estimated using temporal information.
- Some variable units and detailed measurement interpretations may require additional confirmation before quantitative comparison.
- The analysis currently uses TruckScenes v1.2-mini. The final processing workflow should be reusable on the complete TruckScenes dataset.